___

# <font color= #8C6B4F> **GANs for Map Generation: Report** </font>
#### <font color= #2E9AFE> `Deep Learning`</font>
<Strong> Sofía Maldonado, Oscar Josué Rocha & Viviana Toledo </Strong>

_07/04/2026._

___

## <font color= #C2A878> **Objective** </font>

This project aims to build a Generative Adversarial Network (GAN) for map generation via style transfer. Using old and fantasy map images as the style, the GAN should be able to **convert satellite images into specific map representations**.

## <font color= #C2A878> **Data** </font>

Multiple datasets were required for the construction of this project, as well as for trial purposes, since the dataset was not crafted nor achieved on the first try. Our aim is to build a style transfer GAN, which needs a training dataset and a style dataset. The methodology and source of each dataset is detailed below.

### <font color= #F5ECD9> &nbsp; • **Satellite Images** </font>

A first set of satellite images was obtained from the `Sentinel2 satellite`, which is an **Earth observation mission** from the Copernicus Programme by the European Space Agency (ESA). It was first launched on June 2015, and the last launch was performed on September 2024. It orbits at an **altitude of 786 km** and a **maximum Ground Sample Distance (GSD) of 60 meters** (distance between pixels), which results in high-context images [1]. 

The script written for downloading Sentinel2 images can be found in [`scripts/sentinel2.py`](../scripts/sentinel2.py). The dataset contains **5,332 images**.

<p align="center">
  <img src="../reports/figures/1.1-data.png" alt="Sentinel2" width="300">
</p>

The second set of satellite images was obtained from `Open Aerial Map (OAM)`, which is an open collection of aerial imagery. Its objective is to _"host and provide access to imagery for humanitarian response and disaster preparedness"_ [2]. OAM is a colaborative effort, and contains a **variety of satellite images related to areas affected by disasters**, resulting in **close-up images** of certain geographical areas. Although the images were more suited for the project, clouds obstructed some views, not all images have a proper quality-control, and some of them are really zoomed-in.

The script written for downloading OAM images can be found in [`scripts/open_aerial_map.py`](../scripts/open_aerial_map.py). The dataset contains **3,237 images**.

<p align="center">
    <img src="../reports/figures/1.2-data.png" alt="Open Aerial Map" width="300">
</p>

Finally, the last set of images was obtained from `Google Maps` satellite view: a **mosaic of aerial photographies at variable altitudes to map and archive the world at large**. A collection of images from 52 countries across the 5 continents were screenshoted, taken from the 3 most populated cities of each country. Although Google's satellite images can be extracted from an API, the API is blocked behind a paywall, limiting downloads and open access to data. For this reason, a more manual approach was taken for data extraction.

The dataset contains **2,697 images**, and was too large to be uploaded.

<p align="center">
    <img src="../reports/figures/1.3-data.png" alt="Google Maps Satellite View" width="300">
</p>

### <font color= #F5ECD9> &nbsp; • **Map Styles** </font>

Of the 3 datasets obtained, each was paired with a specific style suited for its image range. The Sentinel satellite was paired with fantasy maps at large scale, OAM was paired with old maps, and Google Maps' imagery was paired with close-up fantasy city maps. Each **set of styles contained 20 images**, which were **handpicked on Google Images**.

<p align="center">
  <span style="display: inline-block; text-align: center; margin-right: 60px;">
    <img src="../reports/figures/1.4-data.png" height="300" />
    <br/>
    <em>Style for Sentinel's images</em>
  </span>

  <span style="display: inline-block; text-align: center; margin-right: 60px;">
    <img src="../reports/figures/1.5-data.webp" height="300" />
    <br/>
    <em>Style for OAM's images</em>
  </span>

  <span style="display: inline-block; text-align: center;">
    <img src="../reports/figures/1.6-data.png" height="300" />
    <br/>
    <em>Style for Google Maps' images</em>
  </span>
</p>

## <font color= #C2A878> **Modeling – CycleGAN** </font>

A `CycleGAN` was chosen for the task at hand because it learns from **unpaired data**, meaning there's no information provided as to which image from domain _A_ matches which from domain _B_. Instead, the model `learns to translate between two domains by capturing their underlying distributions and assuming there's a relationship between them`.

The model defines two mappings, such that transformations from $G_A: A \rightarrow B$ and $G_B: B \rightarrow A$ should **approximately reconstruct the original image**. In essence, `it maps an image to itself via an intermediate representation`, or a translation to another domain:

$$
a \rightarrow G_A(a) \rightarrow G_B(G_A(a)) \approx a
$$

CycleGAN implements a **_cyle consistency loss_** as well as an **_adversarial loss_** between image translations to ensure quality. The former prevents the learned mappings $G_A$ and $G_B$ from contradicting each other (the translations), while the latter represents the Discriminators $D_A$ and $D_B$, tasked with ensuring that the generated images are as close to the original distribution as possible. Such that the model's overall loss function is defined as follows [3]: 

$$
\begin{aligned}

\mathcal{L} (G_A, G_B, D_A, D_B) = 
& \mathcal{L}_{\mathrm{GAN}} (G_A, D_B, A, B) \\
& + \mathcal{L}_{\mathrm{GAN}} (G_B, D_A, B, A) \\
& + \lambda \mathcal{L}_{\mathrm{cyc}} (G_A, G_B) \\

\end{aligned}
$$

where:

- The first loss function evaluates transformations $G_A:A \rightarrow B,$
- The second loss function evaluates transformations $G_B:B \rightarrow A,$
- And the third loss function enforces **cycle consistency**, so that mappings don't contradict each other.

Taking into account that our datasets and styles are not paired between each other, and our objective is to **translate images from one domain (satellite images) to another (map representations)**, we imported the CycleGAN model proposed by the authors of the aforementioned paper and creators of the model, available in their GitHub repository [4]. On its own, CycleGAN is incredibly powerful to learn unpaired image representations. Thus, we **trained 3 models for each style using 5 epochs**, achieving great results.

## <font color= #C2A878> **Results** </font>

For exploratory and practicality reasons, **two branches were created to store the models**, and will remain there for isolation. The model for OAM data was stored in `poder_computacional` [`./maps_test`](https://github.com/ch0fas/gans_for_maps/tree/poder_computacional/checkpoints/maps_test), while Google Maps and Sentinel2 models' were stored on `gl_imgs`, under the names [`./maps_test`](https://github.com/ch0fas/gans_for_maps/tree/gl_imgs/checkpoints/maps_test) and [`./maps_test_2`](https://github.com/ch0fas/gans_for_maps/tree/gl_imgs/checkpoints/maps_test_2), respectively.

### <font color= #F5ECD9> &nbsp; • **Sentinel2** </font>

The model trained for Sentinel2 centers around **large-scale images with more landscape context**, creating zoomed-out maps that represent the overall geographical shape of an image. However, it suffers from hallucinations in order to achieve the "island look" a lot of fantasy maps have in common:  

<p align="center">
    <img src="../reports/figures/2.1-results.png" width="370">
    <img src="../reports/figures/2.2-results.png" width="370">
    <img src="../reports/figures/2.3-results.png" width="370">
    <img src="../reports/figures/2.4-results.png" width="370">
</p>

### <font color= #F5ECD9> &nbsp; • **OAM** </font>

OAM's model centers around **_close-up images with not many details_**, creating maps that capture the distribution of cities, with an old worn-out look:

<p align="center">
    <img src="../reports/figures/2.5-results.png" width="370">
    <img src="../reports/figures/2.6-results.png" width="370">
    <img src="../reports/figures/2.7-results.png" width="370">
</p>

### <font color= #F5ECD9> &nbsp; • **Google Maps** </font>

Finally, Google Maps' model focuses on **close-up images with detail**, giving a more polished look to the maps and allowing for greater stylization:

<p align="center">
    <img src="../reports/figures/2.8-results.png" width="370">
    <img src="../reports/figures/2.9-results.png" width="370">
    <img src="../reports/figures/2.10-results.png" width="370">
    <img src="../reports/figures/2.11-results.png" width="370">
</p>

## <font color= #C2A878> **Challenges** </font>

The main challenge we faced while developing this project was **finding suitable images and styles that matched our vision**, since there are multiple approaches you can take for _"converting satellite images to map representations"_, as showcased in the results section. Partly, this is the reason why we have 3 different datasets, each with their own style. On the modeling front, the code for the CycleGAN [4] was fairly easy to comprehend and implement, to the point no changes needed to be made to get it working.

## <font color= #C2A878> **Applications** </font>

## <font color= #C2A878> **Bibliography and References** </font>

[1] Sentinel-2. (2026, 1 de febrero). _Wikipedia_. https://en.wikipedia.org/wiki/Sentinel-2

[2] Open Aerial Map Docs. (2021, 2 de julio). _OpenAerialMap Ecosystem_. https://docs.openaerialmap.org/ecosystem/

[3] Zhu, J.-Y., Park T., Isola, P. & Efros, A. A. (2017). _Unpaired Image-to-Image Translation using Cycle-Consistent Adversarial Networks_. https://arxiv.org/pdf/1703.10593

[4] Zhu, J.-Y., Park T., Isola, P. & Efros, A. A. (2025). _pytorch-CycleGAN-and-pix2pix_. GitHub. https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix